In [1]:
import pygsti
from pygsti.extras import ml
from pygsti.circuits import Circuit
from pygsti.processors.processorspec import QubitProcessorSpec as QPS
from matplotlib import pyplot as plt
import numpy as np
%load_ext autoreload
%autoreload 2

I0000 00:00:1780442750.136447   24994 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780442750.137113   24994 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780442750.199835   24994 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780442751.659213   24994 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
num_qubits = 4

gate_names = ['Gxpi2', 'Gypi2', 'Gcphase']

availability = {'Gcphase': [(0,1), (1,2), (2,3), (3,0)]} # This specifies what qubits a gate can act on. For example, the above says that the Gcphase gate can only act on qubits 0 and 1, or 1 and 2, etc. If a gate is not specified in this dictionary, it is assumed to be available on all qubits (e.g. Gxpi2 and Gypi2 are available on all qubits).

qubit_labels = [i for i in range(num_qubits)]

pspec = QPS(num_qubits = num_qubits, qubit_labels=qubit_labels,
            gate_names=gate_names, availability=availability)

circuit = Circuit('[Gypi2:3Gypi2:1]Gcphase:0:1[Gxpi2:0Gypi2:1Gcphase:2:3]@(0,1,2,3)')

all_gates = []
for gate in pspec.gate_names:
    all_gates += list(pspec.available_gatelabels(gate, pspec.qubit_labels))

encoder = ml.encoding.StandardCircuitEncoder(pspec)

assert(encoder.depth(0) == 0)
assert(encoder.depth(10) == 10)
assert(encoder.gate_indexing == all_gates)

correct_encoding = np.array([[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], 
                             [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
                             [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]])

assert(np.allclose(encoder.layer_encoding(circuit.layer(0)), correct_encoding[0,:]))
assert(np.allclose(encoder.layer_encoding(circuit.layer(1)), correct_encoding[1,:])) 
assert(np.allclose(encoder.layer_encoding(circuit.layer(2)), correct_encoding[2,:]))
assert(np.allclose(encoder(circuit), correct_encoding))

circuits = [circuit,circuit]

correct_circuits_tensor = np.zeros((2, 3, len(all_gates)), float)
correct_circuits_tensor[0, :, :] = correct_encoding
correct_circuits_tensor[1, :, :] = correct_encoding

circuits_tensor = ml.encoding.circuits_to_tensor(circuits, encoder, encoding_depth=None)
assert(np.allclose(circuits_tensor, correct_circuits_tensor))

assert([1, 5, 8, 9] == encoder.indices_for_qubits((1,)))
assert([0, 4, 8, 11] == encoder.indices_for_qubits((0,)))
assert([0, 1, 4, 5, 8, 9, 11] == encoder.indices_for_qubits((1, 0)))

#nbit_strings = [ '0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', 
#                 '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111']
#
#model = pygsti.models.create_crosstalk_free_model(pspec)
#probs = model.probabilities(circuit)
#correct_probabilities = [probs[bs] for bs in nbit_strings]

# Produced using the above code
correct_probabilities = np.array([0., 0., 0., 0., 0.25, 0.25, 0., 0., 0., 0., 0., 0., 0.25, 0.25, 0., 0.])

modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]

tensors = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='matrix')
indices = tensors['indices']
signs = tensors['signs']
probabilities = tensors['probabilities']
alphas = tensors['alphas']

In [3]:
assert(indices.shape == (len(circuits), circuit.depth, len(modelled_error_generators)))
assert(signs.shape == (len(circuits), circuit.depth, len(modelled_error_generators)))
assert(probabilities.shape == (len(circuits), 2 ** num_qubits))

assert(np.allclose(probabilities[0,:], correct_probabilities))

index_0 = ml.errgentools.error_generator_index('H',('ZIII',))
index_1 = ml.errgentools.error_generator_index('S',('IIIZ',))
index_2 = ml.errgentools.error_generator_index('S',('IXII',))

# This error generator is unchanged by any of the layers in the circuit
assert(np.allclose(indices[0,:,1], index_1))
assert(np.allclose(signs[0,:,1], 1))

# The Gxpi2:0 gate in the last layer transforms H_{ZIII} to -H_{YIII}
index_0_transformed = ml.errgentools.error_generator_index('H',('YIII',))
assert(np.allclose(indices[0,0,0], index_0_transformed))
assert(np.allclose(signs[0,0,0], -1))
assert(np.allclose(indices[0,1,0], index_0_transformed))
assert(np.allclose(signs[0,1,0], -1))

# The error generators for the last layer should always be unchanged
assert(np.allclose(indices[0,-1,0], index_0))
assert(np.allclose(indices[0,-1,1], index_1))
assert(np.allclose(signs[0,-1,0], 1))
assert(np.allclose(signs[0,-1,1], 1))

# S_{IXII} should have an effect b/c the marginal distribution on the 2nd qubit is
# a definite outcome (1), and this element of the `alpha` matrix should have been
# filled in as it occurs as and end-of-circuit-error-generator.
assert(alphas[0,-3,index_2] == -0.25)

In [4]:
pspec_2 = QPS(num_qubits = 2, qubit_labels=[i for i in range(2)],
            gate_names=gate_names, availability={'Gcphase': [(0,1),]})

circuit_2 = Circuit('[Gxpi2:0Gypi2:1]@(0,1)')

alphas_2 = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=None)
alphas_2_reduced = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=[4, 25, 30])

# Should be equal for all the error generators we've populated it for.
assert(np.all(alphas_2[:,4] == alphas_2_reduced[:,4]))
assert(np.all(alphas_2[:,25] == alphas_2_reduced[:,25]))
assert(np.all(alphas_2[:,30] == alphas_2_reduced[:,30]))

circuit_3 = Circuit('[Gcphase:0:1]@(0,1)')
alphas_3 = ml.encoding.dense_alpha_matrix(circuit_3, num_qubits=2, populate_for_error_generators=None)
# This is a definite-outcome circuit so has no sensitivity to Hamiltonian parameters at first-order
assert(np.allclose(0., alphas_3[:,0:4**2]))

alphas_2_reduced_added = ml.encoding.dense_alpha_matrix(circuit_2, num_qubits=2, populate_for_error_generators=[10,],
                                               existing_alpha_matrix=alphas_2_reduced)

# Should not have over-written the terms previously computed
assert(np.all(alphas_2_reduced_added[:,4] == alphas_2_reduced[:,4]))
# Should have added in the terms for the 10th error generator
assert(np.all(alphas_2_reduced_added[:,10] == alphas_2[:,10]))

In [5]:
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)

# All indices associated with gates within 1 step on the graph, which for qubit 0 is qubits {0, 1, 3}, and so on.
correct_snipper = [[0, 1, 3, 4, 5, 7, 8, 9, 10, 11], [0, 2, 3, 4, 6, 7, 8, 9, 10, 11], [0, 1, 2, 4, 5, 6, 8, 9, 10, 11]]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
assert(correct_snipper == snipper)

# All indices associated with gates within 0 steps on the graph, which for qubit 0 is just qubit 0, and so on.
correct_snipper = [[0, 4, 8, 11], [3, 7, 10, 11], [1, 5, 8, 9]]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=0)
assert(correct_snipper == snipper)

In [6]:
# Testing succesfully creation of a QPANN
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, signs, indices, alphas, probabilities]
qpann_1 = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='expanded')
output_1 = qpann_1(x)
weights_1 = qpann_1.weights
assert(output_1.shape == (2, 16))

E0000 00:00:1780442752.957264   24994 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
tensors_2 = ml.encoding.error_generator_tensors(circuits, modelled_error_generators, pspec, alpha_representation='concise')
indices_2 = tensors_2['indices']
signs_2 = tensors_2['signs']  # Not needed by the QPANN but still returned
probabilities_2 = tensors_2['probabilities'] # Not needed by the QPANN but still returned
alphas_2 = tensors_2['alphas']

100%|██████████| 2/2 [00:00<00:00, 925.38it/s]


In [8]:
# Testing succesfully creation of a QPANN
adjacency_matrix = ml.snippers.undirected_adjacency_matrix_from_edges(availability['Gcphase'], qubit_labels)
modelled_error_generators = [('H',('ZIII',)),('S',('IIIZ',)), ('S',('IXII',))]
snipper = ml.snippers.layer_snipper_from_qubit_graph(modelled_error_generators, encoder, adjacency_matrix, hops=1)
x = [circuits_tensor, alphas_2, probabilities_2]
qpann_2 = ml.qpanns.QPANN(encoder.length, modelled_error_generators, snipper, probability_computation='concise')
# Call it once to initalize the weights
_ = qpann_2(x)
# Set the weights to be the same as the other QPANN so that we should get the same output
qpann_2.set_weights(weights_1)
output_2 = qpann_2(x)
assert(output_2.shape == (2, 16))
assert(np.allclose(output_1, output_2))

In [9]:
def up_to_weight_k_paulis_from_qubit_graph(
    k: int, n: int, qubit_graph_laplacian: np.ndarray, num_hops: int
) -> list[str]:
    """
    Enumerate Pauli strings of weight 1..k whose *support set* of non-identity qubits
    is connected under a "within num_hops" connectivity rule derived from the qubit graph.

    Convention used throughout this module:
        qubit i  <->  string position (n - 1 - i)
    so qubit 0 is the rightmost character in the Pauli string.

    Parameters
    ----------
    k : int
        Maximum Pauli weight.
    n : int
        Number of qubits (string length).
    qubit_graph_laplacian : numpy.ndarray
        Laplacian matrix of the qubit connectivity graph.
    num_hops : int
        Hop distance defining which qubit pairs are considered "close enough."

    Returns
    -------
    list[str]
        All Pauli strings of weight 1..k with support on qubits connected within num_hops.

    Notes
    -----
    This is a generalization of the legacy k<=2 implementation:
      - For w=1, every single-qubit Pauli is allowed.
      - For w=2, a pair (i,j) is allowed iff i and j are within num_hops hops.
      - For w>2, a support S is allowed iff the induced subgraph on S is connected
        using the "within num_hops" adjacency.
    """
    # ---- Basic input checks ----
    if not isinstance(k, int) or k < 1:
        raise TypeError("Pauli weight, k, must be an integer > 1.")
    if not isinstance(n, int) or n < 0:
        raise TypeError("n must be a non-negative integer.")

    if k > n:
        print("Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)")

    # Clamp k so we never ask for weight > n.
    k = min(k, n)

    # Make a private copy to avoid mutating the caller's matrix accidentally.
    L = np.array(qubit_graph_laplacian, copy=True)

    # Sanity-check dimensions: Laplacian must be n x n.
    if L.shape != (n, n):
        raise ValueError("qubit_graph_laplacian must have shape (n, n).")

    # ---- Build the "within num_hops" connectivity relation ----
    #
    # Legacy code used:
    #   laplace_power = L**num_hops
    #   nodes are considered "within hops" if laplace_power[i,j] != 0
    #
    M = np.linalg.matrix_power(L, num_hops)

    # Remove diagonal entries; we don't want to treat i as "connected to itself"
    # for the purpose of defining edges between *distinct* qubits.
    np.fill_diagonal(M, 0)

    # Convert to a boolean adjacency: close[i,j] == True means "i and j are within num_hops"
    # according to the matrix-power criterion above.
    close = (np.abs(M) > 0)

    # If the underlying graph is undirected, "close" should be symmetric.
    # This line enforces symmetry just in case numerical issues or input oddities break it.
    close = np.logical_or(close, close.T)

    # ---- Helper: test whether a proposed support set is connected ----
    def support_is_connected(support_qubits: tuple[int, ...]) -> bool:
        """
        Return True iff the induced subgraph on `support_qubits` is connected,
        where edges are given by `close`.
        """
        # Empty support shouldn't appear here (we generate weights starting at 1),
        # and a single vertex is trivially connected.
        if len(support_qubits) <= 1:
            return True

        # We'll do a simple DFS/BFS over the support set using the `close` adjacency.
        support = list(support_qubits)
        support_set = set(support)

        # Start from the first qubit in the support.
        seen = {support[0]}
        stack = [support[0]]

        # Standard depth-first traversal restricted to nodes in the support.
        while stack:
            u = stack.pop()

            # Consider only neighbors v that are:
            #   (1) in the support set
            #   (2) adjacent to u under "close"
            #   (3) not yet visited
            for v in support:
                if v not in seen and close[u, v]:
                    seen.add(v)
                    stack.append(v)

        # Connected iff we reached every vertex in the support.
        return len(seen) == len(support_set)

    # ---- Enumerate all valid Pauli strings ----
    base = ['I'] * n            # template for fast construction
    paulis = []                 # output list of Pauli strings
    qubits = range(n)           # qubit labels 0..n-1

    # Enumerate all weights w = 1..k
    for w in range(1, k + 1):

        # Enumerate all possible supports (sets of qubits) of size w.
        for support_qubits in _itertools.combinations(qubits, w):

            # Skip supports that are not connected under the within-num_hops rule.
            if not support_is_connected(support_qubits):
                continue

            # For each support, assign an X/Y/Z to each qubit in the support.
            # There are 3^w assignments.
            for letters in _itertools.product("XYZ", repeat=w):

                # Start from all-identity string, then overwrite the support positions.
                s = base[:]

                # Place each chosen letter at the appropriate *string* index.
                # Remember: qubit q corresponds to string position (n-1-q).
                for q, P in zip(support_qubits, letters):
                    s[n - 1 - q] = P

                # Convert list-of-chars to a string and store it.
                paulis.append("".join(s))

    return paulis

In [10]:
def up_to_weight_k_paulis_from_qubit_graph_old(k: int, n: int, qubit_graph_laplacian: np.array, num_hops: int) -> list:
    """Enumerate Pauli strings of weight 1..k (i.e., all Paulis contain at least one and
    at most k non-identity Paulis) with support on qubits connected by m 
    hops in the qubit connectivity graph.

    Weight-1 Paulis are always included. Weight-2 Paulis are included only when their
    two non-identity qubits are within `num_hops` steps as determined from powers of the
    (provided) graph Laplacian.

    Parameters
    ----------
    k : int
        Maximum Pauli weight (only `k <= 2` supported).
    n : int
        Number of qubits.
    qubit_graph_laplacian : numpy.ndarray
        Laplacian matrix of the qubit connectivity graph.
    num_hops : int
        Hop distance defining which qubit pairs are considered "close enough."

    Returns
    -------
    list[str]
        Pauli strings of length `n`. (Ordering uses reverse indexing convention in this file.) [qubit n, qubit n-1, ..., qubit 0]

    Notes
    -----
    Assumes the graph is connected (not strictly enforced by code).
    """
    assert (k <= 2), "Only implemented up to k = 2!"
    paulis = []

    # weight 1
    for i in range(n):
        for p in ['X', 'Y', 'Z']:
            nq_pauli = n * ['I']
            nq_pauli[n - 1 - i] = p # reverse indexing
            paulis.append(''.join(nq_pauli))
    
    # weight 2
    if k > 1:
        qubit_graph_laplacian = _copy.deepcopy(qubit_graph_laplacian) # Don't delete! Otherwise this function modifies the laplacian globally for some reason?
        laplace_power = np.linalg.matrix_power(qubit_graph_laplacian, num_hops)
        for i in np.arange(n):
            laplace_power[i, i] = 0
        # assert (laplace_power == 0).all(axis=1).any() == False, 'Graph must be connected'
    
        nodes_within_hops = []
        for i in range(n):
            nodes_within_hops.append(np.arange(n)[abs(laplace_power[i, :]) > 0])
    
        for i , qubit_list in enumerate(nodes_within_hops):
            unseen_qubits = qubit_list[np.where(qubit_list > i)[0]]
            for j in unseen_qubits:
                for p in ['X', 'Y', 'Z']:
                        for q in ['X', 'Y', 'Z']:
                            nq_pauli = n * ['I']
                            nq_pauli[n - 1 - i] = p # reverse indexing
                            nq_pauli[n - 1 - j] = q # reverse indexing
                            paulis.append(''.join(nq_pauli))
    
    return paulis

In [11]:
def test_compare_up_to_weight_k_paulis_from_qubit_graph():
    """
    Compare a "new/general" implementation of up_to_weight_k_paulis_from_qubit_graph
    against the legacy implementation (which only supports k<=2) over a variety of inputs.

    How to use
    ----------
    1) Keep the legacy function accessible under a different name, e.g.
         up_to_weight_k_paulis_from_qubit_graph_old = up_to_weight_k_paulis_from_qubit_graph
       before you overwrite it with the new version.

    2) Run this test after defining the new version.

    What this test checks
    ---------------------
    * Only compares k in {1,2} because the legacy code asserts k<=2 and (as written)
      does not include weight-0 identity.
    * Compares outputs as sets (ordering-independent).
    * Tries multiple graphs, sizes, hop counts.

    Important caveat
    ----------------
    The "legacy" implementation determines "within num_hops" using matrix powers of the Laplacian.
    The new implementation must use the *same criterion* for the test to pass (the version I
    provided does).
    """
    # --------------------
    # Helper: build Laplacians for a few standard undirected graphs
    # --------------------
    def line_graph_laplacian(n: int) -> np.ndarray:
        A = np.zeros((n, n), dtype=int)
        for i in range(n - 1):
            A[i, i + 1] = 1
            A[i + 1, i] = 1
        D = np.diag(A.sum(axis=1))
        return D - A

    def complete_graph_laplacian(n: int) -> np.ndarray:
        A = np.ones((n, n), dtype=int) - np.eye(n, dtype=int)
        D = np.diag(A.sum(axis=1))
        return D - A

    def star_graph_laplacian(n: int, center: int = 0) -> np.ndarray:
        A = np.zeros((n, n), dtype=int)
        for j in range(n):
            if j == center:
                continue
            A[center, j] = 1
            A[j, center] = 1
        D = np.diag(A.sum(axis=1))
        return D - A

    # --------------------
    # Helper: compare ordering-independent and provide useful diffs
    # --------------------
    def assert_same_outputs(new_out, old_out, context: str):
        new_set = set(new_out)
        old_set = set(old_out)
        if new_set != old_set:
            only_in_new = sorted(new_set - old_set)[:20]
            only_in_old = sorted(old_set - new_set)[:20]
            raise AssertionError(
                f"Mismatch for {context}\n"
                f"  len(new)={len(new_out)} len(old)={len(old_out)}\n"
                f"  only_in_new (first 20)={only_in_new}\n"
                f"  only_in_old (first 20)={only_in_old}\n"
            )

    # --------------------
    # Test grid
    # --------------------
    ns = [1, 2, 3, 4, 5, 6]
    ks = [1, 2]          # legacy function supports only k<=2 and excludes weight-0 identity
    hops_list = [1, 2, 3]

    laplacians = [
        ("line", line_graph_laplacian),
        ("complete", complete_graph_laplacian),
        ("star", star_graph_laplacian),
    ]

    # --------------------
    # Run tests
    # --------------------
    # NOTE: rename `up_to_weight_k_paulis_from_qubit_graph_old` if your legacy function
    # has a different name.
    for n in ns:
        for lap_name, lap_fn in laplacians:
            L = lap_fn(n)

            for k in ks:
                for num_hops in hops_list:
                    context = f"n={n}, graph={lap_name}, k={k}, hops={num_hops}"

                    old_out = up_to_weight_k_paulis_from_qubit_graph_old(k, n, L, num_hops)
                    new_out = up_to_weight_k_paulis_from_qubit_graph(k, n, L, num_hops)

                    assert_same_outputs(new_out, old_out, context)

    print("PASS: new and old up_to_weight_k_paulis_from_qubit_graph match for all tested inputs.")

In [14]:
import numpy as np
import itertools as _itertools
import copy as _copy
import warnings as _warnings
test_compare_up_to_weight_k_paulis_from_qubit_graph()

Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
Pauli weight cannot exceed the number of qubits. Automatically setting k = min(k,n)
PASS: new and old up_to_weight_k_paulis_from_qubit_graph match for all tested inputs.


In [16]:
# --------------------
# Helper: build Laplacians for a few standard undirected graphs
# --------------------
def line_graph_laplacian(n: int) -> np.ndarray:
    A = np.zeros((n, n), dtype=int)
    for i in range(n - 1):
        A[i, i + 1] = 1
        A[i + 1, i] = 1
    D = np.diag(A.sum(axis=1))
    return D - A

def complete_graph_laplacian(n: int) -> np.ndarray:
    A = np.ones((n, n), dtype=int) - np.eye(n, dtype=int)
    D = np.diag(A.sum(axis=1))
    return D - A

def star_graph_laplacian(n: int, center: int = 0) -> np.ndarray:
    A = np.zeros((n, n), dtype=int)
    for j in range(n):
        if j == center:
            continue
        A[center, j] = 1
        A[j, center] = 1
    D = np.diag(A.sum(axis=1))
    return D - A

up_to_weight_k_paulis_from_qubit_graph(k=2, n=4, qubit_graph_laplacian=line_graph_laplacian(4), num_hops=3)

['IIIX',
 'IIIY',
 'IIIZ',
 'IIXI',
 'IIYI',
 'IIZI',
 'IXII',
 'IYII',
 'IZII',
 'XIII',
 'YIII',
 'ZIII',
 'IIXX',
 'IIYX',
 'IIZX',
 'IIXY',
 'IIYY',
 'IIZY',
 'IIXZ',
 'IIYZ',
 'IIZZ',
 'IXIX',
 'IYIX',
 'IZIX',
 'IXIY',
 'IYIY',
 'IZIY',
 'IXIZ',
 'IYIZ',
 'IZIZ',
 'XIIX',
 'YIIX',
 'ZIIX',
 'XIIY',
 'YIIY',
 'ZIIY',
 'XIIZ',
 'YIIZ',
 'ZIIZ',
 'IXXI',
 'IYXI',
 'IZXI',
 'IXYI',
 'IYYI',
 'IZYI',
 'IXZI',
 'IYZI',
 'IZZI',
 'XIXI',
 'YIXI',
 'ZIXI',
 'XIYI',
 'YIYI',
 'ZIYI',
 'XIZI',
 'YIZI',
 'ZIZI',
 'XXII',
 'YXII',
 'ZXII',
 'XYII',
 'YYII',
 'ZYII',
 'XZII',
 'YZII',
 'ZZII']